# Riesgo de Crédito — Banco Alemán
## Análisis Exploratorio y Modelo Predictivo

**Dataset**: 1.000 solicitudes de préstamos de un banco alemán (German Credit Dataset, Prof. Hofmann, 1994).
Con 20 variables socioeconómicas y crediticias, representativas del tipo de datos que maneja cualquier banco hoy.

**Objetivo**: Reproducir el criterio del banco para aceptar o rechazar solicitudes, construyendo un modelo predictivo
que sea a la vez preciso e interpretable.

**Dos desafíos centrales**:
1. **Muestra pequeña para la cantidad de variables** → la selección de features es crítica antes de modelar.
2. **Costo asimétrico**: clasificar un mal pagador como bueno cuesta 5 veces más que el error inverso.
   Esto obliga a ir más allá del accuracy e incorporar la función de costo en la evaluación.

**Estructura del notebook**:
1. Carga y preparación de datos
2. Análisis Univariado (cada variable por separado)
3. Análisis Bivariado (cada variable vs. el target)
4. Correlaciones entre variables numéricas
5. Information Value (IV) y Weight of Evidence (WoE) — selección de variables
6. Preprocesamiento y split train/validación/test
7. Modelo: Regresión Logística con regularización L1
8. Optimización del threshold con costo asimétrico
9. Interpretabilidad: Odds Ratios
10. Scorecard: puntaje crediticio estandarizado

---
## Sección 1 — Carga y Preparación de Datos

**Para qué**: Cargar el dataset, asignar nombres legibles a las columnas (que en el original son códigos como A11, A32),
recodificar el target en formato binario 0/1 y verificar la calidad del dato (nulos, duplicados, tipos).

**Qué esperamos**: Un dataset limpio de 1.000 filas, 20 variables bien tipadas y el target correctamente identificado.
Si hay nulos o duplicados, habría que decidir cómo tratarlos antes de seguir.

**Nota para otros proyectos**: Este bloque es prácticamente idéntico en cualquier proyecto de clasificación.
La secuencia siempre es: cargar → renombrar → recodificar target → verificar calidad.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve)
import warnings
warnings.filterwarnings('ignore')   # Silencia advertencias de versiones de sklearn

# Configuración global de gráficos — aplicar una vez al inicio es buena práctica
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False    # Saca el borde superior de todos los gráficos
plt.rcParams['axes.spines.right'] = False  # Saca el borde derecho
sns.set_palette('Set2')                    # Paleta de colores amigable con daltónicos
print('Librerías cargadas.')

Librerías cargadas.


In [ ]:
# El dataset tiene columnas con nombres numéricos (1, 2, 3...) en lugar de nombres descriptivos.
# El diccionario col_names mapea cada posición a un nombre semántico usando german_clean.docx como referencia.
df_raw = pd.read_excel('Base_Clientes_Alemanes.xlsx', sheet_name=0, header=0)

col_names = {
    df_raw.columns[0]:  'cliente_id',
    df_raw.columns[1]:  'cuenta_corriente',      # Saldo en cuenta corriente (A11=<0 DM, A12=0-200, A13=>=200, A14=sin cuenta)
    df_raw.columns[2]:  'duracion_meses',         # Duración del crédito en meses
    df_raw.columns[3]:  'historial_credito',      # Comportamiento crediticio pasado (A30-A34)
    df_raw.columns[4]:  'proposito',              # Para qué se pide el crédito (auto, muebles, etc.)
    df_raw.columns[5]:  'monto_credito',          # Monto en Deutsche Marks (DM)
    df_raw.columns[6]:  'ahorros',                # Saldo en cuenta de ahorros (A61=<100, A65=sin cuenta)
    df_raw.columns[7]:  'empleo_desde',           # Antigüedad laboral (A71=desempleado, A75=>=7 años)
    df_raw.columns[8]:  'tasa_cuota',             # % del ingreso que representa la cuota (1-4)
    df_raw.columns[9]:  'estado_civil_sexo',      # Sexo y estado civil combinados
    df_raw.columns[10]: 'otros_deudores',         # Si hay co-deudor o garante
    df_raw.columns[11]: 'anios_residencia',       # Años en la residencia actual (1-4)
    df_raw.columns[12]: 'propiedad',              # Tipo de propiedad que posee
    df_raw.columns[13]: 'edad',                   # Edad en años
    df_raw.columns[14]: 'otros_planes_cuota',     # Otros planes de cuota activos (banco, comercios, ninguno)
    df_raw.columns[15]: 'vivienda',               # Tipo de vivienda (alquiler, propia, gratis)
    df_raw.columns[16]: 'num_creditos_banco',     # Número de créditos activos en este banco
    df_raw.columns[17]: 'tipo_trabajo',           # Tipo de trabajo (calificado, no calificado, etc.)
    df_raw.columns[18]: 'num_dependientes',       # Personas a cargo
    df_raw.columns[19]: 'telefono',               # Si tiene teléfono registrado
    df_raw.columns[20]: 'trabajador_extranjero',  # Si es trabajador extranjero
    df_raw.columns[21]: 'target'                  # Variable objetivo: 1=bueno, 2=malo
}
df = df_raw.rename(columns=col_names).copy()

# Recodificación del target: el dataset original usa 1=bueno, 2=malo.
# Convertimos a 0=bueno (no mora), 1=malo (mora) — convención estándar en clasificación binaria.
# El filtro por isin([1,2]) descarta cualquier valor corrupto que no sea 1 o 2.
df = df[df['target'].isin([1, 2])].copy()
df['target'] = (df['target'] == 2).astype(int)   # True/False → 1/0

# Separar variables numéricas y categóricas para tratarlas diferente en el preprocesamiento
num_cols = ['duracion_meses', 'monto_credito', 'tasa_cuota',
            'anios_residencia', 'edad', 'num_creditos_banco', 'num_dependientes']

cat_cols = ['cuenta_corriente', 'historial_credito', 'proposito', 'ahorros',
            'empleo_desde', 'estado_civil_sexo', 'otros_deudores', 'propiedad',
            'otros_planes_cuota', 'vivienda', 'tipo_trabajo', 'telefono',
            'trabajador_extranjero']

print(f'Shape: {df.shape}')
print(f'Target -> 0 (bueno): {(df.target==0).sum()} | 1 (malo): {(df.target==1).sum()}')

Shape: (1000, 22)
Target -> 0 (bueno): 700 | 1 (malo): 300


**Resultado**: El dataset contiene exactamente **1.000 registros** sin valores inválidos en el target.
- **700 clientes buenos (70%)** y **300 malos (30%)** — desbalance 70/30.
- Los valores categóricos son códigos internos del banco: A11 = cuenta corriente con saldo negativo (<0 DM),
  A14 = sin cuenta corriente, A32 = historial de crédito al día, etc. Estos códigos se decodifican en las secciones siguientes.
- El target se recodificó de 1/2 a 0/1 porque sklearn y la mayoría de las librerías esperan que la clase positiva
  (el evento de interés, en este caso la mora) sea el 1.

**Diagnóstico inicial**: un modelo "ingenuo" que apruebe a todos tendría 70% de accuracy y un costo de
300 × 5 = **1.500 unidades**. Ese es nuestro baseline real a superar.

In [ ]:
# head() muestra las primeras filas — sirve para confirmar que los nombres de columnas quedaron bien
# y que los valores tienen sentido antes de seguir.
print('=== Primeras filas ===')
display(df.head())

# dtypes dice el tipo de dato de cada columna: int64 para numéricas, object para texto (categóricas).
# Si una numérica quedara como object, habría que convertirla antes de modelar.
print('\n=== Tipos de datos ===')
display(df.dtypes.to_frame('tipo').T)

nulos = df.isnull().sum()
print('\n=== Nulos por columna ===')
print(nulos[nulos > 0] if nulos.any() else 'Sin valores nulos — dataset completo.')

print(f'\nDuplicados: {df.duplicated().sum()}')

**Resultado**: El dataset está en excelente estado:
- **Sin valores nulos** en ninguna de las 22 columnas → no hace falta estrategia de imputación.
- **Sin duplicados** → cada fila es un cliente distinto.
- Las 7 variables numéricas tienen tipo `int64` (correcto).
- Las 13 categóricas tienen tipo `object` — esto significa que son strings como "A11", "A32", etc.
  En la Sección 6 las convertiremos a números mediante one-hot encoding para que el modelo pueda usarlas.

**Nota para otros proyectos**: cuando hay nulos, la decisión más común es imputar con la mediana (numéricas)
o con la moda (categóricas). En credit scoring a veces el nulo tiene significado propio (cliente sin datos =
mayor riesgo) y se lo trata como una categoría aparte.

In [ ]:
# describe() calcula las estadísticas descriptivas clásicas: count, mean, std, min, Q1, Q2(mediana), Q3, max.
# Es el primer vistazo cuantitativo a las distribuciones.
print('=== Estadísticas descriptivas — Variables numéricas ===')
display(df[num_cols].describe().round(2))

**Resultado — Lo que dicen los números**:

| Variable | Media | Mediana | Diferencia | Interpretación |
|---|---|---|---|---|
| monto_credito | 3.271 DM | 2.320 DM | +41% | Sesgo derecho fuerte: pocos créditos muy grandes suben la media |
| duracion_meses | 20.9 | 18.0 | +16% | La mayoría son créditos cortos, pero hay casos hasta 72 meses |
| edad | 35.5 | 33.0 | +8% | Población joven-adulta, concentrada entre 25-45 años |
| tasa_cuota | 2.97 | 3.00 | ~0 | Variable discreta 1-4; distribución casi uniforme |
| anios_residencia | 2.84 | 3.00 | ~0 | Variable discreta 1-4; muy poca variabilidad continua |
| num_creditos_banco | 1.41 | 1.00 | — | Más del 70% tiene un solo crédito activo |
| num_dependientes | 1.16 | 1.00 | — | Casi todos tienen 1 dependiente; rango 1-2 |

**Señal importante**: cuando media > mediana, hay sesgo positivo (cola derecha). En `monto_credito` la diferencia
del 41% indica que los outliers de montos altos jalan la media hacia arriba, pero no representan a la mayoría.
`tasa_cuota` y `anios_residencia` con rango 1-4 tendrán poca variabilidad → anticipamos IVs bajos.

In [ ]:
# Gráfico de la distribución del target: barplot para ver los conteos absolutos,
# pie chart para ver la proporción relativa.
conteo = df['target'].value_counts().sort_index()
colores = ['#66b2b2', '#e07b7b']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

bars = axes[0].bar(['Bueno (0)', 'Malo (1)'], conteo.values, color=colores, edgecolor='white', width=0.5)
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 8, str(v), ha='center', fontweight='bold', fontsize=13)
axes[0].set_title('Distribución del Target', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Cantidad de clientes')
axes[0].set_ylim(0, 820)

axes[1].pie(conteo.values, labels=['Bueno (0)', 'Malo (1)'], colors=colores,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proporción Good / Bad', fontweight='bold', fontsize=13)

plt.suptitle('Variable Target — Riesgo de Crédito', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Resultado**: Desbalance **70% buenos / 30% malos** confirmado visualmente.

**¿Por qué importa este desbalance?**
- Un clasificador que siempre diga "bueno" tiene 70% de accuracy **sin aprender nada**.
- Para el banco, ese clasificador tendría un costo = 300 × 5 = **1.500** (todos los malos entran en mora).
- Por eso usamos dos mecanismos:
  1. `class_weight='balanced'` en el modelo: le dice a sklearn que pese más los errores en la clase minoritaria.
  2. Métricas más informativas: AUC-ROC, PR-AUC y **función de costo asimétrico**.

**Nota para otros proyectos**: desbalances hasta 80/20 son manejables con `class_weight='balanced'`.
Con desbalances extremos (99/1, como fraude bancario) se usan técnicas adicionales como SMOTE o undersampling.

---
## Sección 2 — Análisis Univariado

**Para qué**: Estudiar cada variable de forma aislada para entender su distribución, detectar outliers
y categorías con muy pocos casos (que pueden causar problemas en el modelado).

**Qué esperamos**: Variables financieras (monto, duración) con distribuciones asimétricas típicas del sector;
variables discretas con poca variabilidad; y algunas categorías con tan pocos casos que deberemos agruparlas.

**Herramientas utilizadas**:
- **Histograma + KDE**: muestra la forma de la distribución. Útil para detectar sesgo (asimetría) y bimodalidad.
- **Boxplot**: muestra mediana, rango intercuartil (IQR) y outliers (puntos fuera de 1.5×IQR).
- **Skewness (sesgo)**: valor positivo = cola derecha; negativo = cola izquierda; ~0 = simétrico.

In [ ]:
# Diccionario de etiquetas descriptivas para los gráficos (evita mostrar nombres de variable cruda)
labels_num = {
    'duracion_meses':     'Duración (meses)',
    'monto_credito':      'Monto del crédito (DM)',
    'tasa_cuota':         'Tasa cuota (% ingreso)',
    'anios_residencia':   'Años en residencia',
    'edad':               'Edad (años)',
    'num_creditos_banco': 'N° créditos en banco',
    'num_dependientes':   'N° dependientes'
}

fig, axes = plt.subplots(len(num_cols), 2, figsize=(13, 4 * len(num_cols)))

for i, col in enumerate(num_cols):
    label = labels_num[col]
    data  = df[col].dropna()

    # Histograma: muestra frecuencia de cada valor
    axes[i, 0].hist(data, bins=30, color='#66b2b2', edgecolor='white')
    # KDE (Kernel Density Estimate): versión suavizada del histograma, muestra la forma continua
    ax2 = axes[i, 0].twinx()   # twinx crea un segundo eje Y en el mismo gráfico
    data.plot.kde(ax=ax2, color='#2e7d7d', linewidth=2)
    ax2.set_ylabel('Densidad', color='#2e7d7d', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='#2e7d7d')
    ax2.set_ylim(bottom=0)
    # Líneas de media y mediana: si están muy separadas, hay sesgo
    axes[i, 0].axvline(data.mean(),   color='#e07b7b', ls='--', lw=1.5, label=f'Media: {data.mean():.1f}')
    axes[i, 0].axvline(data.median(), color='#f4a261', ls=':',  lw=1.5, label=f'Mediana: {data.median():.1f}')
    axes[i, 0].set_title(f'{label} — Histograma', fontweight='bold')
    axes[i, 0].set_ylabel('Frecuencia')
    axes[i, 0].legend(fontsize=8)

    # Boxplot: la caja representa el 50% central de los datos (IQR = Q3 - Q1)
    # Los bigotes llegan hasta 1.5×IQR; los puntos fuera son outliers por esta definición
    axes[i, 1].boxplot(data, vert=False, patch_artist=True,
                       boxprops=dict(facecolor='#66b2b2', color='#2e7d7d'),
                       medianprops=dict(color='#e07b7b', linewidth=2),
                       whiskerprops=dict(color='#2e7d7d'),
                       capprops=dict(color='#2e7d7d'),
                       flierprops=dict(marker='o', color='#e07b7b', alpha=0.4, markersize=4))
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    n_out = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    axes[i, 1].set_title(f'{label} — Boxplot', fontweight='bold')
    axes[i, 1].set_xlabel(f'Skew: {data.skew():.2f}  |  Outliers (IQR×1.5): {n_out}')
    axes[i, 1].set_yticks([])

plt.suptitle('Análisis Univariado — Variables Numéricas', fontsize=15, fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()

**Resultado — Variables numéricas**:

- **monto_credito** (skew=1.950, 72 outliers): la distribución más sesgada del dataset.
  El valor máximo (18.424 DM) es 8 veces la mediana (2.320 DM). Esto es normal en datos crediticios:
  la mayoría de los préstamos son pequeños, pero los grandes elevan la media.
- **duracion_meses** (skew=1.094, 70 outliers): similar patrón. La mayoría son créditos a 12-24 meses,
  pero hay casos extremos de hasta 72 meses (6 años) que generan la cola derecha.
- **edad** (skew=1.021, 23 outliers): concentración entre 25-45 años. Los casos de más de 60 años
  son los "outliers", pero no son errores — son clientes reales menos frecuentes.
- **tasa_cuota** y **anios_residencia**: rango 1-4, distribución casi uniforme.
  Su boxplot casi no tiene caja visible → poca variabilidad real.
- **num_dependientes**: los "155 outliers" son un **artefacto estadístico**, no datos erróneos.
  Como solo toma los valores 1 y 2, el IQR es tan pequeño que el valor 2 queda formalmente fuera del bigote.

**Implicación práctica**: el sesgo de monto y duración no es un problema para la regresión logística
con scaling (el StandardScaler lo normaliza). Para modelos basados en árboles (Random Forest, XGBoost),
el sesgo tampoco importa porque no asumen distribución normal.

In [ ]:
# Diccionario de etiquetas legibles para cada código categórico
# Fuente: german_clean.docx (decodificador oficial del dataset)
cat_labels = {
    'cuenta_corriente':      {'A11': '<0 DM',       'A12': '0-200 DM', 'A13': '>=200 DM',      'A14': 'Sin cuenta'},
    'historial_credito':     {'A30': 'Sin créd.',   'A31': 'Pago banco','A32': 'Pago OK',       'A33': 'Mora pasada', 'A34': 'Crítico'},
    'proposito':             {'A40': 'Auto nuevo',  'A41': 'Auto usado','A42': 'Muebles',       'A43': 'TV/Radio',
                              'A44': 'Electrodom.', 'A45': 'Reparac.',  'A46': 'Educación',
                              'A48': 'Recapacit.',  'A49': 'Negocio',   'A410': 'Otros'},
    'ahorros':               {'A61': '<100 DM',     'A62': '100-500',   'A63': '500-1000',      'A64': '>=1000', 'A65': 'Sin ahorros'},
    'empleo_desde':          {'A71': 'Desempleado', 'A72': '<1 año',    'A73': '1-4 años',      'A74': '4-7 años', 'A75': '>=7 años'},
    'estado_civil_sexo':     {'A91': 'H divorc.',   'A92': 'M divorc.', 'A93': 'H soltero',    'A94': 'H casado',  'A95': 'M soltera'},
    'otros_deudores':        {'A101': 'Ninguno',    'A102': 'Co-solid.','A103': 'Garante'},
    'propiedad':             {'A121': 'Inmueble',   'A122': 'Seg.vida', 'A123': 'Auto/otros',  'A124': 'Sin prop.'},
    'otros_planes_cuota':    {'A141': 'Banco',      'A142': 'Comercios','A143': 'Ninguno'},
    'vivienda':              {'A151': 'Alquiler',   'A152': 'Propia',   'A153': 'Gratis'},
    'tipo_trabajo':          {'A171': 'Desempl.',   'A172': 'No calif.','A173': 'Calificado',  'A174': 'Gerente'},
    'telefono':              {'A191': 'No',         'A192': 'Sí'},
    'trabajador_extranjero': {'A201': 'Sí',         'A202': 'No'}
}

nrows_plot = (len(cat_cols) + 1) // 2
fig, axes = plt.subplots(nrows_plot, 2, figsize=(15, 4 * nrows_plot))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    conteo_cat = df[col].value_counts().sort_index()
    etiquetas  = cat_labels.get(col, {})
    conteo_cat.index = [etiquetas.get(str(k), str(k)) for k in conteo_cat.index]

    bars = axes[i].bar(conteo_cat.index, conteo_cat.values,
                       color=sns.color_palette('Set2', len(conteo_cat)), edgecolor='white')
    # Etiqueta con conteo absoluto y porcentaje sobre cada barra
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2, h + 2,
                     f'{int(h)}\n({h/len(df)*100:.0f}%)',
                     ha='center', va='bottom', fontsize=7.5)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    axes[i].set_ylabel('Frecuencia')
    axes[i].set_ylim(0, conteo_cat.max() * 1.25)
    axes[i].tick_params(axis='x', rotation=25)

# Ocultar el último panel si el número de categóricas es impar
if len(cat_cols) % 2 != 0:
    axes[-1].set_visible(False)

plt.suptitle('Análisis Univariado — Variables Categóricas', fontsize=15, fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()

**Resultado — Variables categóricas**:

- **cuenta_corriente**: la más heterogénea. 394 clientes sin cuenta (39.4%), 274 con saldo negativo (27.4%),
  269 con 0-200 DM (26.9%) y solo 63 con saldo positivo alto (6.3%). Alta variación entre categorías.
- **historial_credito**: 530 clientes con historial al día (A32 = 53%), pero todas las categorías tienen
  representación suficiente. Variable con buena cobertura.
- **estado_civil_sexo**: 548 hombres solteros (55%). La categoría A95 (mujer soltera) tiene **0 casos** →
  desaparecerá automáticamente al hacer one-hot encoding.
- **trabajador_extranjero**: 963/1000 son extranjeros (96.3%) → casi sin variación. Esta variable
  probablemente no discriminará nada porque casi todos tienen el mismo valor.
- **tipo_trabajo**: distribución bastante uniforme entre las 4 categorías → también sospechoso de bajo poder.
- **otros_deudores** y **telefono**: alta concentración en una sola categoría (>85%) → baja variabilidad.

**Lo que anticipa este análisis**: las variables con distribuciones muy concentradas (>90% en una categoría)
como `trabajador_extranjero` y `telefono` probablemente tendrán IV bajo en la Sección 5 y serán descartadas.

---
## Sección 3 — Análisis Bivariado

**Para qué**: Estudiar la relación de **cada variable con el target** (bueno/malo). Queremos identificar
qué variables discriminan mejor entre buenos y malos pagadores antes de modelar.

**Herramientas**:
- **Boxplot por clase + Test de Mann-Whitney** (numéricas): Mann-Whitney es el equivalente no paramétrico
  del t-test. No asume distribución normal, por eso es más adecuado para datos financieros sesgados.
  Un p-value < 0.05 indica que la diferencia entre buenos y malos es estadísticamente significativa.
- **Bad Rate por categoría + Test Chi-cuadrado** (categóricas): el bad rate es la proporción de malos
  dentro de cada categoría. El Chi-cuadrado mide si la asociación con el target es significativa.

**Qué esperamos**: cuenta_corriente, historial_credito, monto y duración deberían mostrar diferencias claras.

In [ ]:
# Boxplots separados por clase para cada variable numérica
# El objetivo es ver visualmente si la distribución de buenos y malos es diferente
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

buenos = df[df['target'] == 0]
malos  = df[df['target'] == 1]

for i, col in enumerate(num_cols):
    data_b = buenos[col].dropna()
    data_m = malos[col].dropna()

    bplot = axes[i].boxplot([data_b, data_m], vert=True, patch_artist=True,
                             labels=['Bueno', 'Malo'],
                             boxprops=dict(facecolor='#66b2b2'),
                             medianprops=dict(color='#e07b7b', linewidth=2),
                             flierprops=dict(marker='o', alpha=0.3, markersize=3))
    bplot['boxes'][1].set_facecolor('#e07b7b')

    # Mann-Whitney U Test: H0 = las distribuciones de buenos y malos son iguales
    # Si rechazamos H0 (p < 0.05), la variable discrimina entre las clases
    stat, p = stats.mannwhitneyu(data_b, data_m, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    axes[i].set_title(f'{labels_num[col]}\np={p:.3f} {sig}', fontweight='bold', fontsize=9)
    axes[i].set_ylabel('Valor')

axes[-1].set_visible(False)
plt.suptitle('Bivariado — Numéricas vs Target (Mann-Whitney)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Resultado — Test de Mann-Whitney (numéricas vs target)**:

| Variable | Significancia | Interpretación |
|---|---|---|
| duracion_meses | *** (p<0.001) | Los malos tienen créditos más largos (mediana buenos: ~18m, malos: ~24m) |
| monto_credito | *** (p<0.001) | Los malos toman montos mayores en promedio |
| edad | ** (p<0.01) | Los malos son en promedio más jóvenes |
| tasa_cuota | * (p<0.05) | Diferencia leve pero estadísticamente real |
| anios_residencia | ns | Sin diferencia significativa entre buenos y malos |
| num_creditos_banco | ns | Sin diferencia — cuántos créditos tenés no predice el riesgo |
| num_dependientes | ns | Sin diferencia — cuántos dependientes tenés tampoco predice |

**Clave para el modelo**: las 3 variables "ns" (no significativas) probablemente tendrán IV bajo
y serán descartadas antes de modelar. Las variables `*`, `**` y `***` son candidatas a entrar.

**Nota para otros proyectos**: Mann-Whitney es la herramienta correcta para variables continuas
con sesgo (como montos y duraciones). Si los datos fueran normales, usaríamos el t-test de Student.

In [ ]:
# Bad rate por categoría: ¿qué proporción de malos tiene cada categoría?
# Esta es la métrica más directa para entender el poder discriminante de una variable categórica.
fig, axes = plt.subplots(nrows_plot, 2, figsize=(15, 4.5 * nrows_plot))
axes = axes.flatten()

bad_rates_resumen = {}

for i, col in enumerate(cat_cols):
    # Calcular el bad rate (media del target) para cada categoría, ordenado de mayor a menor riesgo
    bad_rate = df.groupby(col)['target'].mean().sort_values(ascending=False)
    counts   = df[col].value_counts()

    etiquetas = cat_labels.get(col, {})
    bad_rate.index  = [etiquetas.get(str(k), str(k)) for k in bad_rate.index]
    counts.index    = [etiquetas.get(str(k), str(k)) for k in counts.index]

    bad_rates_resumen[col] = bad_rate

    # Colorear por nivel de riesgo: rojo ≥40%, naranja 25-40%, verde <25%
    colores_bar = ['#e07b7b' if v >= 0.4 else '#f4a261' if v >= 0.25 else '#66b2b2'
                   for v in bad_rate.values]
    bars = axes[i].bar(bad_rate.index, bad_rate.values, color=colores_bar, edgecolor='white')

    for bar, v in zip(bars, bad_rate.values):
        n = counts.get(bad_rate.index[list(bad_rate.values).index(v)], 0)
        axes[i].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                     f'{v:.0%}\n(n={n})', ha='center', va='bottom', fontsize=7.5)

    # Línea de referencia: bad rate global del dataset (30%)
    axes[i].axhline(df['target'].mean(), color='gray', ls='--', lw=1,
                    label=f'Promedio global {df["target"].mean():.0%}')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    axes[i].set_ylabel('Bad Rate')
    axes[i].set_ylim(0, 1.0)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(axis='x', rotation=30)

    # Chi-cuadrado: H0 = la variable y el target son independientes
    # Si p < 0.05, hay asociación estadística → la variable es útil
    ct = pd.crosstab(df[col], df['target'])
    _, p_chi, _, _ = stats.chi2_contingency(ct)
    sig = '***' if p_chi < 0.001 else '**' if p_chi < 0.01 else '*' if p_chi < 0.05 else 'ns'
    axes[i].set_xlabel(f'chi² p={p_chi:.3f} {sig}', fontsize=8)

if len(cat_cols) % 2 != 0:
    axes[-1].set_visible(False)

plt.suptitle('Bivariado — Categóricas: Bad Rate por Categoría', fontsize=15, fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()

**Resultado — Bad rate por categoría (Chi-cuadrado)**:

- **cuenta_corriente (***) — Mayor spread del dataset**: saldo negativo (<0 DM) → bad rate ~50%;
  sin cuenta corriente → bad rate ~12%. Diferencia de ~38 puntos porcentuales entre extremos.
  Una variable con tanto spread suele dominar el modelo.
- **historial_credito (***) — Spread de 45%**: mora pasada (A33) → 40% de malos; historial crítico (A34) → 20%.
  Paradoja aparente: el historial "crítico" tiene menos mora actual que el de "mora pasada".
  Posible explicación: clientes con historial crítico que igual les prestaron son los que renegociaron bien.
- **ahorros (***)**: sin ahorros → ~35%; ahorros >=1000 DM → ~14%. La capacidad de ahorro predice el riesgo.
- **tipo_trabajo (p=0.597, ns)**: el tipo de trabajo no discrimina nada. Los 4 tipos tienen bad rate similar.
- **telefono (ns)** y **trabajador_extranjero (ns)**: sin asociación con el target.

**Para llevar a otros proyectos**: el "spread" (diferencia entre el bad rate más alto y el más bajo de una variable)
es un indicador rápido del poder discriminante. Spread > 15 puntos = variable interesante. Spread < 5 = descartar.

---
## Sección 4 — Correlaciones entre Variables Numéricas

**Para qué**: Detectar pares de variables numéricas altamente correlacionadas entre sí (multicolinealidad).
Si dos variables dicen básicamente lo mismo, agregar ambas al modelo no suma información pero sí ruido.

**Umbral de preocupación**: |r| > 0.70 indica correlación alta. Entre 0.30 y 0.70 es moderada — no elimina
ninguna variable por sí sola, pero es información útil para interpretar el modelo.

**Nota para otros proyectos**: la multicolinealidad afecta más a la regresión logística que a los modelos
de árboles. Con L1 (Lasso), el modelo puede manejarla solo reduciendo coeficientes a cero.

In [ ]:
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
# mask=triu oculta el triángulo superior para no mostrar la misma correlación dos veces
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=[labels_num[c] for c in num_cols],
            yticklabels=[labels_num[c] for c in num_cols],
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación — Variables Numéricas', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Imprimir los pares con correlación notable (|r| > 0.30) para análisis textual
print('Pares con |correlación| > 0.30:')
for col1 in num_cols:
    for col2 in num_cols:
        if col1 < col2:   # Evitar duplicados y auto-correlación
            r = corr_matrix.loc[col1, col2]
            if abs(r) > 0.3:
                print(f'  {labels_num[col1]} — {labels_num[col2]}: r={r:.3f}')

**Resultado**:

- **Duración vs Monto: r = 0.625** — La correlación más alta del dataset. Tiene sentido económico:
  los préstamos de mayor monto se devuelven en más tiempo. No llega a 0.70, por lo que no eliminaremos ninguna.
- El resto de los pares tienen correlaciones bajas (|r| < 0.30) → sin multicolinealidad problemática.

**Conclusión práctica**: podemos usar todas las variables numéricas sin preocuparnos por redundancia.
La regularización L1 del modelo sabrá reducir el coeficiente de la que aporte menos información marginal.

**Nota importante**: esta matriz solo analiza variables numéricas. Para detectar redundancia entre categóricas
habría que usar Chi-cuadrado o Cramér's V, pero con el IV que calculamos en la siguiente sección es suficiente.

---
## Sección 5 — Information Value (IV) y Weight of Evidence (WoE)

**Para qué**: El IV es la métrica estándar en credit scoring para cuantificar el poder predictivo de cada
variable. A diferencia del análisis bivariado (que da significancia estadística), el IV da una escala
comparable entre variables numéricas y categóricas para rankearlas y filtrarlas.

**Interpretación del IV** — umbrales de la industria:
| IV | Interpretación |
|---|---|
| < 0.02 | No predictiva — descartar |
| 0.02 – 0.10 | Predictor débil — incluir con cautela |
| 0.10 – 0.30 | Predictor moderado — incluir |
| 0.30 – 0.50 | Predictor fuerte — priorizar |
| > 0.50 | Sospechoso — posible data leakage |

**¿Cómo funciona el WoE?**: para cada bin/categoría de una variable, calcula:
`WoE = ln(% de Buenos en ese bin / % de Malos en ese bin)`
- WoE > 0: ese bin tiene más buenos que malos → señal de bajo riesgo
- WoE < 0: ese bin tiene más malos → señal de alto riesgo
- WoE = 0: ese bin tiene la misma proporción que el promedio

**El IV suma** la contribución de cada bin ponderada por cuánto difieren sus distribuciones:
`IV = Σ (Dist_Buenos_i − Dist_Malos_i) × WoE_i`

**Nota para otros proyectos**: el IV+WoE es la pipeline estándar en modelos de scoring bancario (Basel II/III).
Se puede aplicar a cualquier clasificación binaria en la que necesites interpretabilidad.

In [ ]:
def calcular_woe_iv_categorica(df, col, target='target', smoothing=0.5):
    '''
    Calcula WoE e IV para una variable categórica.

    Parámetros clave:
    - smoothing=0.5 (Laplace): suma 0.5 a numerador y denominador de cada bin para evitar
      log(0) cuando una categoría tiene 0 buenos o 0 malos en la muestra.
      Sin esto, una sola categoría con todos malos haría el IV infinito.
    - Umbral del 3%: categorías con menos del 3% de observaciones se agrupan en "OTRO".
      En 1.000 registros esto equivale a <30 casos — demasiado pocos para calcular WoE estable.
    '''
    total_b = (df[target] == 0).sum()
    total_m = (df[target] == 1).sum()

    # Identificar categorías raras (< 3% del total) y agruparlas en OTRO
    freq = df[col].value_counts(normalize=True)
    raras = freq[freq < 0.03].index
    col_tmp = df[col].apply(lambda x: 'OTRO' if x in raras else x)

    tabla = pd.DataFrame({'cat': col_tmp, 'target': df[target]})
    grp = tabla.groupby('cat')['target'].agg(['count', 'sum'])
    grp.columns = ['total', 'malos']
    grp['buenos'] = grp['total'] - grp['malos']

    # dist_b y dist_m: ¿qué % del total de buenos/malos cae en este bin?
    # Ej: si hay 700 buenos y 200 están en una categoría → dist_b = 200/700 = 28.6%
    grp['dist_b'] = (grp['buenos'] + smoothing) / (total_b + smoothing * len(grp))
    grp['dist_m'] = (grp['malos']  + smoothing) / (total_m + smoothing * len(grp))

    grp['woe'] = np.log(grp['dist_b'] / grp['dist_m'])     # WoE del bin
    grp['iv']  = (grp['dist_b'] - grp['dist_m']) * grp['woe']   # Contribución al IV
    grp['bad_rate'] = grp['malos'] / grp['total']

    return grp, grp['iv'].sum()   # Retorna la tabla y el IV total de la variable


def calcular_woe_iv_numerica(df, col, target='target', n_bins=5, smoothing=0.5):
    '''
    Calcula WoE e IV para una variable numérica.

    Usa pd.qcut (quantile cut): divide el rango en n_bins grupos con igual número
    de observaciones (~200 por bin en 1.000 registros con 5 bins).
    Esto es mejor que pd.cut (igual ancho) porque cada bin tiene suficiente muestra.
    '''
    total_b = (df[target] == 0).sum()
    total_m = (df[target] == 1).sum()

    tmp = df[[col, target]].copy().dropna()
    # duplicates='drop': si hay muchos valores repetidos, algunos bins pueden colapsar
    tmp['bin'] = pd.qcut(tmp[col], q=n_bins, duplicates='drop')

    grp = tmp.groupby('bin')[target].agg(['count', 'sum'])
    grp.columns = ['total', 'malos']
    grp['buenos'] = grp['total'] - grp['malos']

    grp['dist_b'] = (grp['buenos'] + smoothing) / (total_b + smoothing * len(grp))
    grp['dist_m'] = (grp['malos']  + smoothing) / (total_m + smoothing * len(grp))

    grp['woe'] = np.log(grp['dist_b'] / grp['dist_m'])
    grp['iv']  = (grp['dist_b'] - grp['dist_m']) * grp['woe']
    grp['bad_rate'] = grp['malos'] / grp['total']

    return grp, grp['iv'].sum()


# Calcular IV para las 20 variables
iv_results = {}
woe_tables = {}

for col in num_cols:
    tabla, iv = calcular_woe_iv_numerica(df, col)
    iv_results[col] = iv
    woe_tables[col] = ('num', tabla)

for col in cat_cols:
    tabla, iv = calcular_woe_iv_categorica(df, col)
    iv_results[col] = iv
    woe_tables[col] = ('cat', tabla)

# Construir tabla resumen ordenada por IV descendente
iv_df = pd.DataFrame({'Variable': list(iv_results.keys()),
                       'IV': list(iv_results.values())}).sort_values('IV', ascending=False)
iv_df['Interpretacion'] = iv_df['IV'].apply(lambda x:
    'Sospechoso' if x > 0.5 else
    'Fuerte'     if x >= 0.3 else
    'Moderado'   if x >= 0.1 else
    'Débil'      if x >= 0.02 else 'No predictiva')

display(iv_df.reset_index(drop=True))

                 Variable        IV Interpretacion
0        cuenta_corriente  0.659056     Sospechoso
1       historial_credito  0.290832       Moderado
2          duracion_meses  0.213164       Moderado
3                 ahorros  0.187938       Moderado
4               proposito  0.152047       Moderado
5               propiedad  0.111799       Moderado
6           monto_credito  0.092127          Débil
7            empleo_desde  0.085656          Débil
8                vivienda  0.083800          Débil
9                    edad  0.067332          Débil
10     otros_planes_cuota  0.058441          Débil
11      estado_civil_sexo  0.044830          Débil
12  trabajador_extranjero  0.039274          Débil
13         otros_deudores  0.031128          Débil
14             tasa_cuota  0.025218          Débil
15           tipo_trabajo  0.009001  No predictiva
16               telefono  0.006282  No predictiva
17     num_creditos_banco  0.002884  No predictiva
18       anios_residencia  0.00

**Resultado — IV de las 20 variables**:

**El hallazgo más importante**: `cuenta_corriente` tiene IV=0.659 → **es excluida** porque supera
el umbral IV_MAX=0.60 (sospecha de data leakage). Aunque es la variable más discriminante del dataset
(spread de 38 puntos en bad rate), la política de la industria establece que IVs tan altos son sospechosos.
En la práctica, esto puede ocurrir cuando la variable "filtra" indirectamente el target (e.g., si el banco
ya tomó decisiones basadas en la cuenta corriente). Conservarla podría generar sobreajuste.

**Variables seleccionadas (14/20)** con IV entre 0.02 y 0.60:
- **5 predictoras moderadas** (IV ≥ 0.10): historial_credito, duracion_meses, ahorros, proposito, propiedad
- **9 predictoras débiles** (0.02 ≤ IV < 0.10): monto_credito, empleo_desde, vivienda, edad, otros_planes_cuota, estado_civil_sexo, trabajador_extranjero, otros_deudores, tasa_cuota

**6 variables descartadas** (IV < 0.02 o > IV_MAX): cuenta_corriente, tipo_trabajo, telefono,
num_creditos_banco, anios_residencia, num_dependientes. Incluirlas solo añadiría ruido al modelo.

**Nota**: las variables con IV=0 o IV≈0 (num_dependientes, anios_residencia) no aportan absolutamente
nada a separar buenos de malos — exactamente lo que predijo el análisis bivariado.

In [ ]:
# Gráfico visual del ranking de IV con líneas de umbral
IV_MIN, IV_MAX = 0.02, 0.60

# Colorear según la interpretación del IV
colores_iv = []
for iv in iv_df['IV']:
    if iv > IV_MAX:    colores_iv.append('#9b2226')  # Sospechoso — rojo oscuro
    elif iv >= 0.3:    colores_iv.append('#2d6a4f')  # Fuerte — verde oscuro
    elif iv >= 0.1:    colores_iv.append('#52b788')  # Moderado — verde claro
    elif iv >= IV_MIN: colores_iv.append('#f4a261')  # Débil — naranja
    else:              colores_iv.append('#cccccc')  # No predictiva — gris

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(iv_df['Variable'], iv_df['IV'], color=colores_iv, edgecolor='white')
ax.axvline(IV_MIN, color='#f4a261', ls='--', lw=1.5, label=f'Umbral mínimo IV={IV_MIN}')
ax.axvline(IV_MAX, color='#9b2226', ls='--', lw=1.5, label=f'Umbral máximo IV={IV_MAX}')

for bar, iv in zip(bars, iv_df['IV']):
    ax.text(iv + 0.005, bar.get_y() + bar.get_height()/2, f'{iv:.3f}', va='center', fontsize=9)

ax.invert_yaxis()   # Variable con mayor IV al tope del gráfico
ax.set_xlabel('Information Value (IV)')
ax.set_title('Ranking de Variables por Information Value', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# Selección final de variables para el modelo
vars_seleccionadas = iv_df[(iv_df['IV'] >= IV_MIN) & (iv_df['IV'] <= IV_MAX)]['Variable'].tolist()
print(f'Variables seleccionadas para el modelo (IV entre {IV_MIN} y {IV_MAX}): {len(vars_seleccionadas)}')
print(vars_seleccionadas)

**Resultado**: El gráfico muestra claramente la frontera de decisión entre variables útiles e inútiles.

**Variables seleccionadas (14)**:
`historial_credito` (0.291), `duracion_meses` (0.213), `ahorros` (0.188), `proposito` (0.152),
`propiedad` (0.112), `monto_credito` (0.092), `empleo_desde` (0.086), `vivienda` (0.084),
`edad` (0.067), `otros_planes_cuota` (0.058), `estado_civil_sexo` (0.045),
`trabajador_extranjero` (0.039), `otros_deudores` (0.031), `tasa_cuota` (0.025).

**Nota para otros proyectos**: aplicar IV es una de las formas más eficientes de reducir dimensionalidad
en credit scoring. Es rápido, transparente y no requiere correr ningún modelo primero.
Los umbrales 0.02 y 0.60 son convenciones de la industria, no reglas matemáticas absolutas.
En otro contexto, podrías ajustarlos (ej. tomar IV_MIN=0.05 si querés ser más estricto).

In [ ]:
# WoE plots para las top 6 variables por IV
# El WoE de cada bin muestra en qué dirección y qué tan fuerte discrimina esa categoría
top6 = iv_df.head(6)['Variable'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(top6):
    tipo, tabla = woe_tables[col]
    tabla_plot = tabla.copy().reset_index()
    x_vals = tabla_plot.iloc[:, 0].astype(str)

    ax1 = axes[i]
    ax2 = ax1.twinx()

    # Barras de WoE: verde = bin con más buenos que malos; rojo = más malos que buenos
    bars = ax1.bar(range(len(x_vals)), tabla_plot['woe'],
                   color=['#66b2b2' if w >= 0 else '#e07b7b' for w in tabla_plot['woe']],
                   edgecolor='white', alpha=0.85, label='WoE')
    # Línea de bad rate: permite ver el riesgo concreto por bin
    ax2.plot(range(len(x_vals)), tabla_plot['bad_rate'],
             color='#2d6a4f', marker='o', lw=2, ms=6, label='Bad Rate')
    ax1.axhline(0, color='gray', ls='--', lw=1)  # Línea de WoE=0: punto neutro

    ax1.set_xticks(range(len(x_vals)))
    ax1.set_xticklabels(x_vals, rotation=30, ha='right', fontsize=8)
    ax1.set_ylabel('WoE', color='#2e7d7d')
    ax2.set_ylabel('Bad Rate', color='#2d6a4f')
    ax2.set_ylim(0, 1)
    axes[i].set_title(f'{col.replace("_", " ").title()}\nIV = {iv_results[col]:.3f}',
                      fontweight='bold', fontsize=10)

plt.suptitle('WoE y Bad Rate — Top 6 Variables por IV', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Resultado — Cómo leer los gráficos de WoE**:

- **Barras verdes** (WoE > 0): ese bin tiene más buenos que el promedio → menor riesgo.
- **Barras rojas** (WoE < 0): ese bin tiene más malos que el promedio → mayor riesgo.
- **Línea verde (bad rate)**: el porcentaje real de malos en ese bin. Siempre se mueve en sentido
  inverso al WoE: a mayor bad rate, menor WoE.

**Patrones observados**:
- **duracion_meses**: monotónico ascendente en bad rate. A mayor plazo del crédito, mayor riesgo.
  Esto es intuitivo: créditos más largos son más difíciles de sostener.
- **historial_credito (A34 = "Crítico")**: WoE positivo y bajo bad rate (~20%), menor que "mora pasada" (~40%).
  Contraintuitivo: clientes catalogados como "críticos" que igualmente tomaron crédito tienen menos mora.
  Probable explicación: son clientes que renegociaron exitosamente deudas anteriores.
- **cuenta_corriente**: ya fue excluida por IV > 0.60, pero su WoE muestra el gradiente más pronunciado.

**Para llevar a otro proyecto**: un WoE monotónico (siempre creciente o siempre decreciente con el riesgo)
es ideal. Indica que la variable captura una relación lineal con el riesgo, lo que hace al modelo más estable.

---
## Sección 6 — Preprocesamiento y Split Train/Validación/Test

**Para qué**: Preparar los datos en el formato que el modelo requiere y dividir la muestra en tres
conjuntos con roles distintos:
- **Train (60%)**: el modelo aprende de estos datos.
- **Validación (20%)**: se usa para optimizar hiperparámetros y el threshold. El modelo no lo "ve" al entrenar.
- **Test (20%)**: se usa **una sola vez** al final para reportar el rendimiento real. No se toca antes.

**¿Por qué 60/20/20 y no 80/20?**: Con 1.000 registros y un modelo paramétrico (regresión logística),
60% de train (600 casos) es suficiente. Tener un set de validación separado del test es crucial para
optimizar el threshold sin contaminar la evaluación final.

**Encoding**: las variables categóricas se convierten a columnas binarias 0/1 (one-hot).
El StandardScaler normaliza las numéricas para que las regularizaciones sean comparables entre variables.

In [ ]:
# Solo usamos las variables seleccionadas por IV — las 14 con mayor poder predictivo
features = vars_seleccionadas
X = df[features].copy()
y = df['target'].copy()

# Separar qué features son numéricas y cuáles categóricas dentro de las seleccionadas
num_sel = [c for c in features if c in num_cols]
cat_sel = [c for c in features if c in cat_cols]

# Imputación preventiva (en este dataset no hay nulos, pero es buena práctica incluirla
# para que el código sea reutilizable en otros datasets)
for c in num_sel:
    X[c] = X[c].fillna(X[c].median())       # Mediana: robusta a outliers
for c in cat_sel:
    X[c] = X[c].fillna('MISSING').astype(str)  # Tratar el nulo como una categoría propia

# One-hot encoding: convierte cada categoría en una columna binaria (0 o 1)
# drop_first=True: elimina la primera categoría de cada variable para evitar la "trampa de la variable dummy"
# (multicolinealidad perfecta: si todas las dummies son 0, la primera categoría es implícita)
X_enc = pd.get_dummies(X, columns=cat_sel, drop_first=True)
X_enc.columns = X_enc.columns.astype(str)

# Split 60 / 20 / 20 estratificado
# stratify=y: garantiza que la proporción 70/30 se mantenga en los 3 conjuntos
# Paso 1: separar el 20% de test
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_enc, y, test_size=0.20, stratify=y, random_state=42)
# Paso 2: del 80% restante, tomar 25% como validación → 0.25 × 0.80 = 0.20 del total
X_train, X_valid, y_train, y_valid = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

print(f'Train: {X_train.shape}  |  Valid: {X_valid.shape}  |  Test: {X_test.shape}')
print(f'Train bad rate: {y_train.mean():.2%}  |  Valid: {y_valid.mean():.2%}  |  Test: {y_test.mean():.2%}')

# StandardScaler: resta la media y divide por la desviación estándar → variables en escala comparable
# with_mean=False: compatibilidad con matrices dispersas (sparse); aquí es buena práctica aunque no sea sparse
# REGLA CRÍTICA: fit SOLO en train → transform en valid y test. Así simulamos producción real.
scaler = StandardScaler(with_mean=False)
X_train_sc = scaler.fit_transform(X_train)   # Aprende la media y std del train
X_valid_sc = scaler.transform(X_valid)       # Aplica la misma transformación sin re-aprender
X_test_sc  = scaler.transform(X_test)        # Ídem para test

Train: (600, 38)  |  Valid: (200, 38)  |  Test: (200, 38)
Train bad rate: 30.00%  |  Valid: 30.00%  |  Test: 30.00%


**Resultado**:
- **Train=600, Valid=200, Test=200** — split 60/20/20 exacto.
- **Bad rate 30.0%** en los 3 conjuntos → la estratificación funcionó perfectamente.
  Sin estratificación, podría haber quedado, por azar, un conjunto con 25% de malos y otro con 35%,
  lo que haría las comparaciones entre sets inconsistentes.
- **38 features** después del one-hot encoding. Las 14 variables categóricas y numéricas seleccionadas
  se expandieron a 38 columnas binarias + continuas.

**Data leakage — el error más común en ML**: si hubiéramos hecho `scaler.fit_transform(X_enc)` antes
de dividir los conjuntos, el scaler habría "visto" los datos de test al aprender la media y std.
El test ya no sería un conjunto completamente desconocido. Siempre: fit en train, transform en todo lo demás.

**Nota para otros proyectos**: en datasets más grandes (>10.000 filas), un split 80/10/10 es más común.
Con muestras pequeñas como esta, 60/20/20 da más datos al scaler y al modelo sin sacrificar demasiada
capacidad de validación.

---
## Sección 7 — Modelo: Regresión Logística con Regularización L1

**Para qué**: La regresión logística con penalización L1 (también llamada Lasso logístico) es el modelo
estándar en credit scoring por tres razones:
1. **Interpretabilidad**: los coeficientes se traducen directamente a Odds Ratios (Sección 9).
2. **Selección automática de variables**: L1 lleva exactamente a cero los coeficientes de features poco útiles.
3. **Robustez con muestras pequeñas**: la regularización penaliza la complejidad, evitando sobreajuste.

**Hiperparámetros clave**:
- `penalty='l1'`: usa regularización L1 (Lasso). L2 (Ridge) encoge coeficientes pero no los anula.
  L1 hace selección automática → mejor para interpretabilidad.
- `C=0.1`: inverso de la fuerza de regularización. C=1 es la default (poca regularización);
  C=0.1 = más regularización = modelo más parsimonioso. Adecuado para muestras de 600 casos.
- `solver='liblinear'`: el único solver de sklearn que soporta L1 de forma eficiente con datos pequeños.
- `class_weight='balanced'`: sklearn calcula automáticamente pesos inversamente proporcionales a la frecuencia
  de cada clase. Con 70/30, asigna peso 1/0.70 ≈ 1.43 a los buenos y 1/0.30 ≈ 3.33 a los malos.
  Esto hace que los errores en la clase minoritaria (malos) pesen más durante el entrenamiento.

In [ ]:
logit = LogisticRegression(
    penalty='l1',            # Lasso: lleva coeficientes irrelevantes exactamente a 0
    solver='liblinear',      # Único solver de sklearn compatible con L1 + muestras pequeñas
    class_weight='balanced', # Compensa el desbalance 70/30 sin modificar los datos
    C=0.1,                   # Regularización fuerte: apropiada para 600 casos de train
    max_iter=500,            # Iteraciones máximas para la convergencia
    random_state=42          # Reproducibilidad
)
logit.fit(X_train_sc, y_train)

# Predecir probabilidades (no clases) — usamos probabilidades para optimizar el threshold después
# [:, 1] selecciona la probabilidad de la clase 1 (malo)
y_prob_valid = logit.predict_proba(X_valid_sc)[:, 1]
y_prob_test  = logit.predict_proba(X_test_sc)[:, 1]

coef_no_nulos = (logit.coef_[0] != 0).sum()
print(f'Modelo entrenado correctamente.')
print(f'Coeficientes no nulos: {coef_no_nulos} / {len(logit.coef_[0])}')
print(f'→ L1 descartó {len(logit.coef_[0]) - coef_no_nulos} features automáticamente.')

Modelo entrenado correctamente.
Coeficientes no nulos: 23 / 38
→ L1 descartó 15 features automáticamente.


**Resultado**: 23 coeficientes no nulos sobre 38 posibles → **L1 descartó 15 features automáticamente**.

Esto significa que de las 38 columnas que entraron al modelo (resultado del one-hot encoding de las 14
variables seleccionadas por IV), el Lasso consideró que 15 no aportan información marginal suficiente
para justificar su inclusión y las anuló.

**¿Por qué esto es bueno?**: un modelo con 23 variables activas es más parsimonioso que uno con 38.
Menos coeficientes = menos riesgo de sobreajuste = mejor generalización a datos nuevos.
También simplifica la interpretación: los Odds Ratios de la Sección 9 solo analizarán estos 23.

**Si quisieras más o menos selección**: aumentar C (ej. C=1.0) → más coeficientes no nulos.
Bajar C (ej. C=0.01) → menos coeficientes no nulos. C es un hiperparámetro que podrías optimizar
con cross-validation, pero con esta muestra pequeña C=0.1 es razonable.

In [ ]:
# Evaluación con threshold=0.50 (el default de sklearn)
# threshold=0.50 significa: si la probabilidad predicha de mora ≥ 50% → clasificar como malo
y_pred_test = (y_prob_test >= 0.5).astype(int)

print('=== Métricas en Test (threshold=0.50) ===')
print(f'Accuracy : {(y_pred_test == y_test).mean():.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_test, y_prob_test):.4f}')
print(f'PR-AUC   : {average_precision_score(y_test, y_prob_test):.4f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['Bueno', 'Malo']))

# Matriz de confusión: muestra los 4 tipos de predicciones posibles
cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Bueno', 'Pred Malo'],
            yticklabels=['Real Bueno', 'Real Malo'])
ax.set_title('Matriz de Confusión — Test (threshold=0.50)', fontweight='bold')
plt.tight_layout()
plt.show()

**Resultado — Evaluación con threshold=0.50**:

- **AUC-ROC = 0.7102**: el modelo supera el umbral mínimo de 0.70 para credit scoring. Esto significa
  que, dado un cliente bueno y uno malo al azar, el modelo rankea correctamente al malo por encima en el 71% de los casos.
- **PR-AUC = 0.5443** vs. baseline de 0.30 (tasa de malos en test) → el modelo es un 81% mejor que
  clasificar aleatoriamente al ordenar por probabilidad de mora.

**Lectura de la matriz de confusión con threshold=0.50**:
| | Pred Bueno | Pred Malo |
|---|---|---|
| **Real Bueno** | TN = 87 | FP = 53 |
| **Real Malo** | FN = 22 | TP = 38 |

- **TN=87**: buenos clasificados correctamente (no problema).
- **FP=53**: buenos rechazados por error (costo=1 cada uno → 53 unidades). El banco pierde un buen cliente.
- **FN=22**: **malos aprobados** (costo=5 cada uno → 110 unidades). El banco otorgó crédito y no cobró.
- **TP=38**: malos detectados correctamente (no problema).

**Costo total con threshold=0.50**: 5×22 + 1×53 = 110 + 53 = **163 unidades**.
Esto ya es mucho mejor que el baseline (1.500), pero el threshold 0.50 no considera la asimetría de costos.
Lo optimizamos en la siguiente sección.

In [ ]:
# Curvas ROC y Precision-Recall — dos perspectivas complementarias del modelo
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
prec, rec, _ = precision_recall_curve(y_test, y_prob_test)
auc_roc = roc_auc_score(y_test, y_prob_test)
auc_pr  = average_precision_score(y_test, y_prob_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Curva ROC: eje X = FPR (qué % de buenos rechazamos), eje Y = TPR (qué % de malos detectamos)
# El área bajo la curva (AUC) resume el rendimiento en todos los thresholds posibles
axes[0].plot(fpr, tpr, color='#2d6a4f', lw=2, label=f'AUC-ROC = {auc_roc:.3f}')
axes[0].plot([0,1], [0,1], 'k--', lw=1, label='Clasificador aleatorio (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2d6a4f')
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR / Recall)')
axes[0].set_title('Curva ROC', fontweight='bold', fontsize=13)
axes[0].legend()

# Curva PR: eje X = Recall (qué % de malos detectamos), eje Y = Precision (de los que dijimos malo, cuántos lo son)
# El baseline es la tasa de malos en el dataset (30%) — un clasificador random estaría en esa línea
axes[1].plot(rec, prec, color='#e07b7b', lw=2, label=f'PR-AUC = {auc_pr:.3f}')
axes[1].axhline(y_test.mean(), color='gray', ls='--', lw=1,
                label=f'Baseline aleatorio ({y_test.mean():.2%})')
axes[1].fill_between(rec, prec, alpha=0.1, color='#e07b7b')
axes[1].set_xlabel('Recall (de todos los malos, cuántos detectamos)')
axes[1].set_ylabel('Precisión (de los que marcamos como malos, cuántos lo son)')
axes[1].set_title('Curva Precision-Recall', fontweight='bold', fontsize=13)
axes[1].legend()

plt.suptitle('Evaluación del Modelo — Curvas ROC y PR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Resultado — Cómo leer las curvas**:

**Curva ROC (izquierda)**:
- AUC-ROC = 0.7102 → el modelo tiene capacidad discriminante real.
- La curva está claramente por encima de la diagonal (clasificador aleatorio).
- Limitación: la curva ROC puede ser optimista cuando hay desbalance de clases.
  No distingue bien entre FP y FN en términos de costo absoluto.

**Curva PR (derecha)**:
- PR-AUC = 0.5443 vs. baseline = 0.30 → el modelo es substancialmente mejor que el azar.
- El trade-off clave: a mayor recall (detectar más malos), menor precisión (más falsos positivos).
  No podemos tener ambos al 100% simultáneamente — el threshold es la palanca para controlar esto.
- **Esta curva es la más relevante** con desbalance de clases, porque el eje Y (precisión) está
  directamente afectado por la proporción de positivos en el dataset.

**Nota para otros proyectos**: si el desbalance es severo (ej. 99/1), la curva PR es la única confiable.
La ROC puede dar AUC cercano a 0.90 mientras el modelo falla casi siempre en detectar la clase minoritaria.

---
## Sección 8 — Optimización del Threshold con Costo Asimétrico

**Para qué**: El threshold por defecto (0.50) no tiene en cuenta que los errores no tienen el mismo costo.
En este banco: clasificar un malo como bueno (FN) cuesta 5 veces más que clasificar un bueno como malo (FP).

**Función de costo**: `Costo = 5 × FN + 1 × FP`

**Intuición del threshold**: al bajar el threshold (ej. de 0.50 a 0.30), el modelo es más conservador:
clasifica como "malo" a cualquier cliente con probabilidad ≥ 30%. Esto reduce los FN (malos que se nos
escapan) pero aumenta los FP (buenos rechazados). Con la función de costo del banco, vale la pena
hacer ese intercambio siempre que se eviten más de 5 FP por cada FN adicional que se genera.

**Proceso**: buscamos el threshold en el set de **validación** (no en test) y reportamos el resultado en test.

In [ ]:
COSTO_FN = 5   # Malo aprobado → crédito impago → costo alto
COSTO_FP = 1   # Bueno rechazado → cliente perdido → costo bajo

def costo_total(y_true, y_prob, threshold):
    '''Calcula el costo asimétrico para un threshold dado.'''
    y_pred = (y_prob >= threshold).astype(int)
    cm_t = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm_t.ravel()
    return COSTO_FN * fn + COSTO_FP * fp

# Probar 91 thresholds distintos entre 0.05 y 0.95
# IMPORTANTE: usar y_prob_valid (no y_prob_test) para encontrar el threshold óptimo.
# Si usáramos el test, estaríamos "ajustando el modelo al test" — data leakage implícito.
thresholds = np.linspace(0.05, 0.95, 91)
costos_valid = [costo_total(y_valid, y_prob_valid, t) for t in thresholds]

threshold_optimo = thresholds[np.argmin(costos_valid)]
print(f'Threshold óptimo (encontrado en validación): {threshold_optimo:.2f}')
print(f'Costo en validación con threshold óptimo : {min(costos_valid)}')
print(f'Costo en validación con threshold=0.50    : {costo_total(y_valid, y_prob_valid, 0.50)}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, costos_valid, color='#2d6a4f', lw=2)
ax.axvline(threshold_optimo, color='#e07b7b', ls='--', lw=2,
           label=f'Threshold óptimo = {threshold_optimo:.2f}')
ax.axvline(0.50, color='gray', ls=':', lw=1.5, label='Threshold default = 0.50')
ax.set_xlabel('Threshold de clasificación')
ax.set_ylabel(f'Costo total ({COSTO_FN}×FN + {COSTO_FP}×FP)')
ax.set_title('Función de Costo Asimétrico por Threshold', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Resultado — Curva de costo por threshold**:

- El **threshold óptimo es 0.26** (encontrado en el set de validación).
- La curva tiene forma de U asimétrica:
  - **Thresholds muy bajos** (ej. 0.05): casi todos se clasifican como malos → FN≈0 pero FP explota.
  - **Thresholds muy altos** (ej. 0.90): casi todos se clasifican como buenos → FP≈0 pero FN explota.
  - **El mínimo se desplaza hacia la izquierda** (0.26 en vez de 0.50) porque FN pesa 5 veces más:
    vale la pena aceptar más FP para evitar FN.

**Nota para otros proyectos**: la ratio costo_FN/costo_FP determina cuánto se desplaza el threshold óptimo
del 0.50. Si FN cuesta 10× más, el threshold óptimo será aún más bajo. Si los costos son iguales,
el óptimo debería estar cerca de 0.50 (ajustado por el desbalance de clases).

In [ ]:
# Evaluación final en test con el threshold óptimo
y_pred_optimo = (y_prob_test >= threshold_optimo).astype(int)

print(f'=== Evaluación en Test con threshold óptimo ({threshold_optimo:.2f}) ===')
print(classification_report(y_test, y_pred_optimo, target_names=['Bueno', 'Malo']))

costo_default  = costo_total(y_test, y_prob_test, 0.50)
costo_opt      = costo_total(y_test, y_prob_test, threshold_optimo)
costo_baseline = COSTO_FN * y_test.sum()   # Costo si aprobamos a todos: solo FN

print(f'Costo baseline (aprobar a todos) : {costo_baseline}  (300 malos × 5)')
print(f'Costo threshold 0.50             : {costo_default}')
print(f'Costo threshold óptimo ({threshold_optimo:.2f})   : {costo_opt}')
print(f'Reducción vs baseline            : {(costo_baseline - costo_opt) / costo_baseline:.1%}')

# Comparación lado a lado de las matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, thresh, title in zip(axes,
                              [0.50, threshold_optimo],
                              ['Threshold = 0.50', f'Threshold óptimo = {threshold_optimo:.2f}']):
    ypred = (y_prob_test >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, ypred)
    sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Bueno', 'Pred Malo'],
                yticklabels=['Real Bueno', 'Real Malo'])
    ax.set_title(title, fontweight='bold')
plt.suptitle('Comparación de Matrices de Confusión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Resultado — Comparación de thresholds en test**:

| Escenario | TN | FP | FN | TP | Costo |
|---|---|---|---|---|---|
| Baseline (aprobar todo) | 140 | 0 | 60 | 0 | **300** |
| Threshold = 0.50 | 87 | 53 | 22 | 38 | **163** |
| **Threshold óptimo = 0.26** | 28 | 112 | 5 | 55 | **137** |

**Interpretación del threshold óptimo (0.26)**:
- **FN pasa de 22 a 5** → detectamos 17 malos adicionales que con threshold 0.50 se escapaban.
  En términos económicos: 17 créditos que habrían entrado en mora y ahora se rechazan.
- **FP pasa de 53 a 112** → rechazamos 59 buenos adicionales. El banco pierde esos clientes,
  pero el costo de esa pérdida (59 × 1 = 59) es menor que el beneficio de evitar 17 FN (17 × 5 = 85).

**Reducción de costo vs. baseline**: (300 − 137) / 300 = **54.3%**.

**Nota práctica**: en un banco real, este tipo de análisis se haría con costos reales en moneda
(ej. pérdida esperada por crédito impago = monto × (1 - recovery rate)). El mismo framework aplica.

---
## Sección 9 — Interpretabilidad: Odds Ratios

**Para qué**: La regresión logística predice el logaritmo de las chances (log-odds) de ser mal pagador.
El Odds Ratio (OR) de una variable es la exponencial de su coeficiente: `OR = exp(coeficiente)`.

**Cómo interpretar el OR**:
- **OR = 1.5**: esa variable aumenta las chances de mora en un 50% (por cada unidad adicional, o al pasar de 0 a 1).
- **OR = 0.7**: esa variable reduce las chances de mora en un 30%.
- **OR = 1.0**: la variable no tiene efecto sobre el riesgo (coeficiente = 0).

**Ventaja clave**: este análisis convierte el modelo en un conjunto de reglas de negocio inteligibles.
El equipo de riesgo del banco puede auditar y cuestionar cada OR sin necesidad de entender
la matemática del modelo. Es el lenguaje en que los modelos de scoring se presentan a los reguladores.

In [ ]:
# Construir tabla de coeficientes y Odds Ratios solo para los no-nulos (los que L1 no eliminó)
coef_df = pd.DataFrame({
    'Feature':     X_train.columns,
    'Coeficiente': logit.coef_[0]
})
# Filtrar solo los coeficientes activos (no nulos)
coef_df = coef_df[coef_df['Coeficiente'] != 0].copy()
# OR = exp(coeficiente): convierte el log-odds a escala multiplicativa
coef_df['Odds_Ratio'] = np.exp(coef_df['Coeficiente'])
# Ordenar por valor absoluto del coeficiente: los más influyentes primero
coef_df = coef_df.reindex(coef_df['Coeficiente'].abs().sort_values(ascending=False).index)

print('=== Top 25 features por magnitud del coeficiente ===')
display(coef_df.reset_index(drop=True).head(25))

# Gráfico: barras horizontales de OR, centradas en 1.0
top_or = coef_df.head(20).sort_values('Odds_Ratio')
# Rojo = aumenta riesgo (OR > 1); verde = reduce riesgo (OR < 1)
colors_or = ['#e07b7b' if v > 1 else '#66b2b2' for v in top_or['Odds_Ratio']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_or['Feature'], top_or['Odds_Ratio'], color=colors_or, edgecolor='white')
ax.axvline(1.0, color='gray', ls='--', lw=1.5, label='OR = 1 (sin efecto)')
ax.set_xlabel('Odds Ratio — exp(coeficiente)')
ax.set_title('Odds Ratios — Top 20 Features activas', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Resultado — Cómo leer los Odds Ratios**:

**Top factores que reducen el riesgo (barras verdes, OR < 1)**:
- **historial_credito_A34** (OR = 0.677): tener historial "crítico" **reduce** las chances de mora en un 32.3%.
  Recordar la paradoja del WoE: estos clientes, a pesar de su historial delicado, pagan mejor.
- Categorías con OR << 1 en proposito e historial representan clientes de menor riesgo según el modelo.

**Top factores que aumentan el riesgo (barras rojas, OR > 1)**:
- **duracion_meses** (OR = 1.441): cada mes adicional de plazo **multiplica** las chances de mora por 1.441.
  Dicho de otro modo: un crédito de 24 meses tiene (1.441)^12 ≈ 120× más chances de mora que uno de 12 meses
  (lo que muestra que el efecto compuesto es muy no-lineal y debería interpretarse con cautela a grandes escalas).
- Categorías de proposito o historial con OR > 1 representan perfiles de mayor riesgo.

**Para presentar en una reunión de negocio**: "Un cliente con historial crítico tiene un 32% menos de
probabilidad de entrar en mora que uno con historial al día, controlando el resto de las variables."
Eso es un OR de 0.677 puesto en lenguaje de negocio.

**Nota para otros proyectos**: los OR son la herramienta de explicabilidad en modelos de scoring.
Si el OR de una variable es 10 o 0.1, preguntarse si tiene sentido de negocio antes de aceptarlo.
Valores extremos pueden indicar sobreajuste o multicolinealidad residual.

---
## Sección 10 — Scorecard: Puntaje Crediticio

**Para qué**: El scorecard transforma la probabilidad de mora del modelo en un puntaje entero
(similar al FICO score de EE.UU. o el puntaje Veraz en Argentina). Esto tiene dos ventajas:
1. **Comunicabilidad**: un número entre 300 y 850 es más fácil de interpretar que "probabilidad de mora = 23.7%".
2. **Integración con políticas de crédito**: el banco puede definir "aprobamos a todos con score > 550"
   sin necesidad de entender el modelo subyacente.

**¿Qué es el PDO?** "Points to Double the Odds" — cuántos puntos debe bajar el score para que las
chances de mora se dupliquen. PDO=20 significa que si el score baja de 600 a 580, las chances de mora
se duplican. Es el parámetro que calibra la "velocidad" de la escala.

**Fórmulas estándar de la industria** (Basel II / credit scoring convencional):
- `Factor = PDO / ln(2)` → conversión de escala logarítmica a puntos
- `Offset = Score_ref − Factor × ln(Odds_ref)` → punto de anclaje de la escala
- `Score = Offset − Factor × log_odds` → score individual (mayor score = mejor pagador)

El signo negativo en la última fórmula garantiza que a mayor log-odds de mora → menor score.

In [ ]:
# Parámetros estándar de la industria crediticia
PDO       = 20   # Puntos para duplicar el odds de mora
SCORE_REF = 600  # Score de referencia: a este puntaje el odds es ODDS_REF
ODDS_REF  = 50   # Odds 50:1 a score=600 → de 51 clientes con ese score, 50 son buenos y 1 malo

# Derivar factor y offset a partir de los parámetros
factor = PDO / np.log(2)
offset = SCORE_REF - factor * np.log(ODDS_REF)

print(f'Factor : {factor:.4f}  (puntos por unidad de log-odds)')
print(f'Offset : {offset:.4f}  (punto de anclaje de la escala)')
print(f'Interpretación: a score=600, las chances son 50 buenos por 1 malo')

def prob_a_score(prob, factor, offset):
    '''
    Convierte una probabilidad de mora en un puntaje crediticio.
    prob: probabilidad de ser malo (entre 0 y 1)
    Retorna: score (mayor = mejor pagador)
    '''
    # np.clip evita log(0) o log(inf) para probabilidades extremas
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    log_odds = np.log(prob / (1 - prob))   # Log-odds de ser MALO
    return offset - factor * log_odds      # A mayor log-odds de mora → menor score

scores_test = prob_a_score(y_prob_test, factor, offset)

print(f'\nScore mínimo en test : {scores_test.min():.1f}  (cliente de muy alto riesgo)')
print(f'Score máximo en test : {scores_test.max():.1f}  (cliente de muy bajo riesgo)')
print(f'Score medio en test  : {scores_test.mean():.1f}')

Factor : 28.8539  (puntos por unidad de log-odds)
Offset : 487.1229  (punto de anclaje de la escala)
Interpretación: a score=600, las chances son 50 buenos por 1 malo

Score mínimo en test : 423.0  (cliente de muy alto riesgo)
Score máximo en test : 571.7  (cliente de muy bajo riesgo)
Score medio en test  : 490.9


In [ ]:
# Histograma de scores por clase real (bueno vs malo)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores_test[y_test == 0], bins=30, alpha=0.7, color='#66b2b2',
        label='Bueno (real)', edgecolor='white')
ax.hist(scores_test[y_test == 1], bins=30, alpha=0.7, color='#e07b7b',
        label='Malo (real)', edgecolor='white')
ax.axvline(SCORE_REF, color='gray', ls='--', lw=1.5, label=f'Referencia = {SCORE_REF}')

# Score de corte equivalente al threshold óptimo encontrado en S8
# Es la traducción del threshold (0.26) al lenguaje de puntajes
score_corte = prob_a_score(np.array([threshold_optimo]), factor, offset)[0]
ax.axvline(score_corte, color='#2d6a4f', ls='-', lw=2,
           label=f'Corte óptimo = {score_corte:.0f} pts')

ax.set_xlabel('Score crediticio (mayor = mejor pagador)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del Score por Clase Real', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

aprobados  = (scores_test > score_corte).sum()
rechazados = (scores_test <= score_corte).sum()
print(f'Score de corte: {score_corte:.0f} puntos')
print(f'Aprobados en test (score > {score_corte:.0f}): {aprobados} clientes ({aprobados/len(scores_test):.1%})')
print(f'Rechazados en test (score ≤ {score_corte:.0f}): {rechazados} clientes ({rechazados/len(scores_test):.1%})')

Score de corte: 517 puntos
Aprobados en test (score > 517): 33 clientes (16.5%)
Rechazados en test (score ≤ 517): 167 clientes (83.5%)


**Resultado — Scorecard**:

- **Factor = 28.8539**: cada vez que el log-odds de mora aumenta en 1 punto, el score baja ~29 puntos.
- **Offset = 487.1229**: punto de calibración matemático para que el score sea 600 cuando el odds es 50:1.
- **Rango del modelo**: 423 a 572 puntos. El rango es relativamente estrecho (149 puntos) porque
  el modelo es moderado (AUC=0.71) — un modelo perfecto tendría un rango mucho más amplio.

**Lectura del histograma**:
- Las distribuciones de buenos (azul) y malos (rojo) se superponen pero se ven separadas.
  Los buenos tienden a concentrarse por encima de ~510; los malos, por debajo.
- **Score de corte = 517 puntos**: equivale al threshold óptimo de 0.26.

**Resultado de política**: con el corte en 517 puntos:
- **33 clientes aprobados (16.5%)** en test — el modelo es muy conservador.
- **167 rechazados (83.5%)** — refleja el threshold bajo (0.26) que prioriza no perder créditos malos.

**Nota para otros proyectos**: en la práctica, el banco ajusta el corte según su apetito de riesgo
y su capacidad operativa. El scorecard es la herramienta; la política de corte es una decisión de negocio.
Podrías tener múltiples zonas: "aprobado automático" (score > 550), "revisión manual" (510-550),
"rechazado automático" (score < 510).